# Regenerators AI Image Detector

This notebook launches the same Gradio application used by the local demo. It does not load or evaluate the reserved test dataset.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Project setup

This cell installs Git LFS, clones the repository when needed, and fetches the deployed checkpoint into the canonical `checkpoints/sid_local_lora/` folder.

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("/content/regenerators_aigc_detector")
!apt-get update -qq
!apt-get install -y -qq git-lfs
!git lfs install
if not PROJECT_ROOT.is_dir():
    !git clone https://github.com/h7karu/regenerators_aigc_detector.git {PROJECT_ROOT}
%cd {PROJECT_ROOT}
!git lfs pull --include="checkpoints/sid_local_lora/sid_local_lora_best.pt"
!python -m pip install -q -r requirements.txt

## Verify the checkpoint

The SID checkpoint is tracked through Git LFS and must appear at the same path used by the local application. The shared app automatically uses the repository's validated five-view trimmed-mean TTA policy.

In [ ]:
CHECKPOINT = PROJECT_ROOT / "checkpoints/sid_local_lora/sid_local_lora_best.pt"
CONFIG = PROJECT_ROOT / "configs/sid_local_lora.yaml"
assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
assert CHECKPOINT.stat().st_size > 100_000_000, f"Git LFS pointer found instead of model: {CHECKPOINT}"

## Launch the shared interface

`share=True` creates a temporary public URL. Disable it when the demonstration is over. Uploaded images are processed by the active Colab runtime.

In [ ]:
from demo_app import APP_CSS, DEMO_THEME, create_demo
demo = create_demo(CHECKPOINT, CONFIG, device="auto")
demo.queue(default_concurrency_limit=1).launch(
    share=True,
    show_error=True,
    theme=DEMO_THEME,
    css=APP_CSS,
)